# Dirac Notation, Decoded

Dirac notation looks intimidating. It is not. It is compact NumPy with stylish brackets. By the end of this notebook you will translate any Dirac expression into code on sight.

**Objectives:**
- Build a one-to-one mental map: Dirac symbol ↔ NumPy expression
- Compute inner products, outer products, and tensor products in both notations
- Read expressions like $\langle 0 | H | + \rangle$ without slowing down

**Reference:** See [`../GUIDE.md`](../GUIDE.md).

<!-- browser-runnable -->


In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

# Our reference vocabulary.
zero  = np.array([1, 0], dtype=complex)
one   = np.array([0, 1], dtype=complex)
plus  = (zero + one)  / np.sqrt(2)
minus = (zero - one)  / np.sqrt(2)
H = (1/np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
I = np.eye(2, dtype=complex)

## 1. Kets and bras

- A **ket** $|\psi\rangle$ is a column vector. In NumPy, a 1-D array works fine.
- A **bra** $\langle \psi |$ is its conjugate transpose — a row vector with   conjugated entries.

Why two names for what is essentially the same data? Because pairing a bra with a ket gives an inner product, which is a number, while pairing a ket with a bra gives an outer product, which is a matrix. The notation makes which is which obvious.

In [ ]:
psi = (1/np.sqrt(2)) * np.array([1, 1j])
print('ket |psi>:', psi)
print('bra <psi|:', psi.conj())

## 2. Inner product $\langle a | b \rangle$

This is `a.conj() @ b`. Returns a single complex number that measures overlap.

In [ ]:
print('<0|0> =', zero.conj() @ zero)        # 1
print('<0|1> =', zero.conj() @ one)         # 0
print('<0|+> =', zero.conj() @ plus)        # 1/sqrt(2)
print('<+|-> =', plus.conj() @ minus)       # 0  — plus and minus are orthogonal

## 3. Applying a gate: $U|\psi\rangle$

This is just matrix-vector multiplication: `U @ psi`.

In [ ]:
print('X|0> =', X @ zero)          # |1>
print('X|1> =', X @ one)           # |0>
print('H|0> =', H @ zero)          # |+>
print('H|+> =', H @ plus)          # |0>  — H is its own inverse

## 4. Sandwiches: $\langle a | U | b \rangle$

Read right-to-left: first apply `U` to `|b>`, then take the inner product with `<a|`. In code: `a.conj() @ (U @ b)` — but `@` is associative, so `a.conj() @ U @ b` works too.

In [ ]:
# Examples
print('<0|H|0> =', zero.conj() @ H @ zero)     # 1/sqrt(2)
print('<1|H|0> =', one.conj()  @ H @ zero)     # 1/sqrt(2)
print('<0|H|+> =', zero.conj() @ H @ plus)     # 1
print('<0|X|0> =', zero.conj() @ X @ zero)     # 0

## 5. Outer products $|a\rangle\langle b|$

These are matrices, not numbers. In NumPy: `np.outer(a, b.conj())`. The **projector onto $|\psi\rangle$** is $|\psi\rangle\langle\psi|$ — used constantly in measurement theory.

In [ ]:
P0 = np.outer(zero, zero.conj())          # |0><0|
print('projector onto |0>:')
print(P0)

# Acting on a state: P0 |+> = |0><0|+> = (1/sqrt(2)) |0>
print('\nP0 |+> =', P0 @ plus)

## 6. Tensor products $|a\rangle \otimes |b\rangle$

Same as notebook 1: `np.kron(a, b)`. Often written without the `⊗` symbol — for example, `|01>` is shorthand for `|0> ⊗ |1>`.

In [ ]:
# |01> in conventional notation, equivalent to np.kron(zero, one)
psi_01 = np.kron(zero, one)
print('|01> =', psi_01)

# A simple entangled state: (|00> + |11>) / sqrt(2)  — the Bell pair
bell = (np.kron(zero, zero) + np.kron(one, one)) / np.sqrt(2)
print('Bell state =', bell)
print('measurement probs:', np.abs(bell)**2)   # 50% |00>, 50% |11>, never |01> or |10>

## 7. The full Rosetta stone

Pin this in your head:

| Dirac | NumPy | Type |
|---|---|---|
| $|0\rangle, |1\rangle$ | `np.array([1,0])`, `np.array([0,1])` | ket (vector) |
| $\langle \psi |$ | `psi.conj()` | bra (row vector) |
| $\langle a | b \rangle$ | `a.conj() @ b` | scalar |
| $\| \psi \|^2 = \langle \psi | \psi \rangle$ | `psi.conj() @ psi` | scalar (real, $\geq 0$) |
| $U|\psi\rangle$ | `U @ psi` | ket |
| $\langle a | U | b \rangle$ | `a.conj() @ U @ b` | scalar |
| $\| a \rangle \langle b \|$ | `np.outer(a, b.conj())` | matrix |
| $\| a \rangle \otimes \| b \rangle$ | `np.kron(a, b)` | longer ket |
| $A \otimes B$ | `np.kron(A, B)` | bigger matrix |
| $U^\dagger$ | `U.conj().T` | matrix |
| Born rule: P(outcome k) | `abs(psi[k])**2` | probability |


## 8. Exercises

Three exercises to lock in the translation table. Each has tiered hints --
expand only what you need -- and a check cell that tells you when you have
it. Worked solutions wait at the bottom of the notebook, but run the checks
before you peek.

### Exercise 1 — Compute a sandwich

Translate $\langle + | H | 0 \rangle$ into NumPy and evaluate it. Predict
the number before you run the cell, then let the check confirm you.

Define `sandwich_value`: the number $\langle + | H | 0 \rangle$.

<details><summary>Hint 1 — nudge</summary>

Section 3 showed what $H|0\rangle$ is. Substitute that into the bracket —
what overlap are you really computing?

</details>
<details><summary>Hint 2 — approach</summary>

Use the $\langle a | U | b \rangle$ row of the Rosetta stone: conjugate
the bra vector, then chain everything with `@`. The vocabulary vectors from
the setup cell are all you need.

</details>

In [ ]:
# Exercise 1: Compute the sandwich <+|H|0>.
# Define: sandwich_value -- the number <+|H|0>.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    _v = complex(np.asarray(sandwich_value).item())
    assert abs(_v) <= 1 + 1e-9, (
        "an overlap of normalized states can never exceed magnitude 1"
    )
    assert abs(_v.imag) < 1e-9, "this particular sandwich comes out real"
    assert abs(_v - 1) < 1e-9, (
        "H sends |0> onto one of the two states in the bracket -- the overlap "
        "should be maximal"
    )

### Exercise 2 — A projector of your own

Section 5 built the projector onto $|0\rangle$. Build the projector onto
$|+\rangle$ and verify it behaves like one: it should leave $|+\rangle$
untouched and send $|-\rangle$ to the zero vector.

Define `P_plus`: the $2 \times 2$ projector matrix onto $|+\rangle$.

<details><summary>Hint 1 — nudge</summary>

A projector onto a state pairs that state's ket with its own bra. Which
Rosetta-stone row turns a ket-bra pair into a matrix?

</details>
<details><summary>Hint 2 — approach</summary>

Mirror the `P0` construction from Section 5, swapping in the state you are
projecting onto (the conjugate goes on the bra side). Then apply your
matrix to `plus` and to `minus` with `@` and compare the results against
what a projector promises.

</details>

In [ ]:
# Exercise 2: Build the projector onto |+>.
# Define: P_plus -- the 2x2 projector matrix onto |+>.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    _P = np.asarray(P_plus)
    assert _P.shape == (2, 2), "a one-qubit projector is a 2x2 matrix"
    assert np.allclose(_P, _P.conj().T), "projectors are Hermitian"
    assert np.allclose(_P @ _P, _P), (
        "projecting twice changes nothing: P @ P should equal P"
    )
    assert abs(np.trace(_P) - 1) < 1e-9, (
        "a projector onto a single state has trace 1"
    )
    assert np.allclose(_P @ plus, plus), "P should leave |+> untouched"
    assert np.allclose(_P @ minus, 0), (
        "P should send the orthogonal state |-> to the zero vector"
    )

### Exercise 3 — Tensor, then Born rule

Build the two-qubit state $|+\rangle \otimes |0\rangle$ and compute its
measurement probabilities as a 4-vector. Which outcomes can actually occur,
and with what probability?

Define `probs_plus0`: the length-4 vector of measurement probabilities of
$|+\rangle \otimes |0\rangle$, ordered
$|00\rangle, |01\rangle, |10\rangle, |11\rangle$.

<details><summary>Hint 1 — nudge</summary>

The second qubit is definitely $|0\rangle$. Of the four basis outcomes,
which are even possible? The first qubit's superposition decides how the
probability splits between them.

</details>
<details><summary>Hint 2 — approach</summary>

Two Rosetta-stone rows, in order: the tensor-product row from Section 6 to
build the 4-component state, then the Born-rule row to turn its amplitudes
into probabilities, elementwise.

</details>

In [ ]:
# Exercise 3: Tensor two qubits and read off measurement probabilities.
# Define: probs_plus0 -- the 4-vector of measurement probabilities of
# |+> (x) |0>, ordered |00>, |01>, |10>, |11>.

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    _p = np.real_if_close(np.asarray(probs_plus0))
    assert _p.shape == (4,), (
        "two qubits have four basis outcomes -- expect a 4-vector"
    )
    assert np.all(_p >= -1e-12), "probabilities are never negative"
    assert abs(_p.sum() - 1) < 1e-9, "probabilities must sum to 1"
    assert _p[1] < 1e-9 and _p[3] < 1e-9, (
        "the second qubit is |0>, so outcomes ending in 1 should be impossible"
    )
    assert abs(_p[0] - 0.5) < 1e-9 and abs(_p[2] - 0.5) < 1e-9, (
        "the first qubit is an equal superposition -- its two readings "
        "should be equally likely"
    )

### Solutions

In [ ]:
# --- Exercise 1 ---
sandwich_value = plus.conj() @ H @ zero
print('<+|H|0> =', sandwich_value)   # 1  (because H|0> = |+>, and <+|+> = 1)

# --- Exercise 2 ---
P_plus = np.outer(plus, plus.conj())
print('P|+> =', P_plus @ plus)    # |+>
print('P|-> =', P_plus @ minus)   # zero vector

# --- Exercise 3 ---
probs_plus0 = np.abs(np.kron(plus, zero)) ** 2
print('probs over (|00>, |01>, |10>, |11>):', probs_plus0)
# 50% |00>, 50% |10> -- the second qubit is always 0, the first is 50/50

**You finished notebook 5.** Move on to [`06-bloch-sphere-playground.ipynb`](06-bloch-sphere-playground.ipynb).